<a href="https://colab.research.google.com/github/SANGHATI23/neuroimaging-fhir-omop-fidelity/blob/main/ICHI2027_Colab_02_Real_Source_Reconciliation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import sys, subprocess
subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'pandas==2.2.3'], check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', '-q', 'install', 'pandas==2.2.3'], returncode=0)

In [17]:
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import hashlib, importlib.metadata, json, os, platform, re, sys
import pandas as pd

UTC_NOW = datetime.now(timezone.utc).isoformat()
RUN_BASE = Path(os.environ.get('ICHI2027_WORK_ROOT', '/content' if Path('/content').is_dir() else str(Path.cwd())))
PROJECT_ROOT = RUN_BASE / 'ichi2027_neuroimaging_phase02'
PRIVATE_INPUT = PROJECT_ROOT / 'private_input'
PRIVATE_OUTPUT = PROJECT_ROOT / 'private_outputs'
SHAREABLE_LOCAL = PROJECT_ROOT / 'shareable_outputs'
for directory in (PRIVATE_INPUT, PRIVATE_OUTPUT, SHAREABLE_LOCAL):
    directory.mkdir(parents=True, exist_ok=True)

EXPECTED = (
    'TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv',
    'TRACE_ADNI1_repeat_pairs_REAL.csv',
    'TRACE_ADNI1_landmark12_REAL.csv',
)
SOURCE_ROLES = dict(zip(EXPECTED, ('longitudinal', 'repeat_pairs', 'landmark12')))
DRIVE_ROOT = Path('/content/drive/MyDrive/NeuroFHIR_ICHI2027')
DRIVE_AVAILABLE = False
try:
    from google.colab import drive
    IS_COLAB = True
except ImportError:
    IS_COLAB = False

if IS_COLAB:
    try:
        drive.mount('/content/drive', force_remount=False)
        DRIVE_AVAILABLE = DRIVE_ROOT.parent.is_dir()
    except Exception as exc:
        print('Google Drive not mounted:', type(exc).__name__)

OUTPUT_DEST = DRIVE_ROOT / 'phase02_outputs' if DRIVE_AVAILABLE else SHAREABLE_LOCAL
OUTPUT_DEST.mkdir(parents=True, exist_ok=True)
print(json.dumps({'phase':'02_real_source_reconciliation','utc':UTC_NOW,
                  'drive_output_enabled':DRIVE_AVAILABLE,'output_dir':str(OUTPUT_DEST)}, indent=2))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
{
  "phase": "02_real_source_reconciliation",
  "utc": "2026-09-26T22:09:01.405576+00:00",
  "drive_output_enabled": true,
  "output_dir": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase02_outputs"
}


In [18]:
def sha256_file(path, block=1024*1024):
    h = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for part in iter(lambda: handle.read(block), b''):
            h.update(part)
    return h.hexdigest()

MISSING_TOKENS = {'', 'na', 'n/a', 'nan', 'none', 'null', 'unknown', 'unavailable', 'not available'}
def norm_header(text):
    return re.sub(r'[^a-z0-9]', '', str(text).strip().lower())

def clean_series(series):
    return series.fillna('').astype(str).str.strip()

def not_missing(series):
    return ~clean_series(series).str.lower().isin(MISSING_TOKENS)

def display_count(value, min_cell=5):
    if value is None:
        return None
    value = int(value)
    return f'<{min_cell}' if 0 < value < min_cell else value

ALIASES = {
    'participant_id': ('participant_id','person_id','patient_id','rid','ptid','subject_id','subject','subjectid','id'),
    'scan_id': ('scan_id','study_id','imageuid','image_uid','studyinstanceuid','study_instance_uid','image_id','imagedataid','scanuid'),
    'series_or_acquisition_id': ('series_id','seriesuid','series_instance_uid','acquisition_id','acquisitionid','seriesinstanceuid'),
    'acquisition_date': ('acquisition_date','scan_date','examdate','exam_date','study_date','image_date','exam_dt','scandate'),
    'timepoint': ('timepoint','visit','viscode','viscode2','visit_code','session','event','timepoint_label'),
    'region': ('region','anatomical_region','structure','brain_region','roi','feature_name'),
    'laterality': ('laterality','hemisphere','side','lr'),
    'volume_value': ('value','volume','volume_mm3','volume_cm3','measurement_value','volumevalue'),
    'volume_unit': ('unit','volume_unit','unit_code','ucum_unit','unitsofmeasure'),
    'baseline_diagnosis': ('diagnosis_at_baseline','baseline_diagnosis','dx_bl','dxbaseline','dx_bl_orig'),
    'extraction_model': ('model_name','software_name','segmentation_software','extractor','processing_software'),
    'model_version': ('model_version','software_version','freesurfer_version','pipeline_version'),
    'qc': ('qc','qc_state','quality_control','qc_status'),
    'trace_state': ('trace_state','reliability_state'),
    'trace_rule_version': ('trace_rule_version','reliability_rule_version'),
    'trace_reference': ('trace_reference_id','trace_reference','calibration_reference_id'),
    'trace_reason': ('trace_reason','reliability_reason','trace_reason_code'),
}

def infer_fields(columns):
    by_norm = {}
    for original in columns:
        by_norm.setdefault(norm_header(original), []).append(str(original))
    found = {}
    for semantic, options in ALIASES.items():
        names = sorted(set(name for alias in options for name in by_norm.get(norm_header(alias), [])))
        status = 'MISSING' if not names else 'EXACTLY_ONE' if len(names) == 1 else 'AMBIGUOUS'
        found[semantic] = {'status':status,'candidates':names}
    return found

def resolved(fields, key):
    item = fields.get(key, {})
    return item['candidates'][0] if item.get('status') == 'EXACTLY_ONE' else None

def wide_mri_headers(headers):
    pattern = re.compile(r'hippocamp|entorhin|middle.?tempor|ventric|intracran|(^|[^a-z])icv([^a-z]|$)', re.I)
    return sorted(str(h) for h in headers if pattern.search(str(h)))

def safe_categories(series, max_categories=24):
    valid = clean_series(series)[not_missing(series)]
    counts = valid.value_counts(dropna=False)
    if len(counts) > max_categories:
        return {'unique_labels':int(len(counts)),'labels_suppressed':'too_many_categories'}
    result = {}
    for key, count in counts.items():
        if len(str(key)) <= 32 and re.fullmatch(r'[A-Za-z0-9 _./:+-]{1,32}', str(key)):
            result[str(key)] = display_count(count)
        else:
            result['nonpublic_label_present'] = True
    return {'unique_labels':int(len(counts)), 'counts_by_label':result}

In [19]:
SEARCH_DIRS = [
    DRIVE_ROOT / 'inputs',
    RUN_BASE,
    RUN_BASE / 'ichi2027_neuroimaging_phase01' / 'authorized_input',
    PRIVATE_INPUT,
]

candidates = {name:[] for name in EXPECTED}
for folder in SEARCH_DIRS:
    if not folder.is_dir():
        continue
    for name in EXPECTED:
        candidate = folder / name
        if candidate.is_file():
            candidates[name].append(candidate)

if IS_COLAB and any(not candidates[name] for name in EXPECTED):
    from google.colab import files
    for wanted in EXPECTED:
        if candidates[wanted]:
            continue
        print('Optional authorized upload:', wanted)
        uploaded = files.upload()
        if not uploaded:
            continue
        for name,data in uploaded.items():
            safe = Path(name).name
            if safe not in SOURCE_ROLES:
                continue
            dest = PRIVATE_INPUT / safe
            if dest.exists() and sha256_file(dest) != hashlib.sha256(data).hexdigest():
                raise RuntimeError(f'Conflicting versions already present: {safe}')
            dest.write_bytes(data)
            candidates[safe].append(dest)
        del uploaded

source_files = {}
file_conflicts = {}
for name, locations in candidates.items():
    unique_locations = list(dict.fromkeys(str(p.resolve()) for p in locations))
    if not unique_locations:
        continue
    fingerprints = {p:sha256_file(p) for p in map(Path, unique_locations)}
    if len(set(fingerprints.values())) != 1:
        file_conflicts[name] = [{'folder':str(p.parent),'sha256':digest} for p,digest in fingerprints.items()]
        continue
    source_files[name] = Path(unique_locations[0])

if file_conflicts:
    (PRIVATE_OUTPUT/'CONFLICTING_INPUTS_PRIVATE.json').write_text(json.dumps(file_conflicts, indent=2))
    raise RuntimeError('STOP: two locations contain different versions of the same source filename. See private conflict report.')
if not source_files:
    raise FileNotFoundError('STOP: none of the three genuine TRACE_ADNI1 CSV files are available.')
print('Actual input files located:',list(source_files))
print('Missing input files:',[name for name in EXPECTED if name not in source_files])

Actual input files located: ['TRACE_ADNI1_longitudinal_COMPLETE_REAL.csv', 'TRACE_ADNI1_repeat_pairs_REAL.csv', 'TRACE_ADNI1_landmark12_REAL.csv']
Missing input files: []


In [20]:
DATA = {}
FINGERPRINTS = {}
SOURCE_PROFILES = {}
ID_SETS = {}

for name in EXPECTED:
    if name not in source_files:
        SOURCE_PROFILES[SOURCE_ROLES[name]]={'file_present':False,'status':'MISSING_REAL_INPUT'}
        continue
    file=source_files[name]
    frame=pd.read_csv(file, dtype=str, keep_default_na=False, encoding='utf-8', on_bad_lines='error', low_memory=False)
    if frame.columns.duplicated().any():
        raise ValueError(f'STOP: duplicate CSV column names: {name}')
    fields = infer_fields(frame.columns)
    role = SOURCE_ROLES[name]
    DATA[role] = frame
    FINGERPRINTS[role] = {'filename':name, 'bytes':file.stat().st_size,'sha256':sha256_file(file)}
    counts = {}
    for semantic in ALIASES:
        matched = resolved(fields, semantic)
        if matched:
            counts[semantic] = {'nonempty_rows':int(not_missing(frame[matched]).sum()),
                                'empty_rows':int((~not_missing(frame[matched])).sum())}
    id_column=resolved(fields,'participant_id')
    ids = set(clean_series(frame[id_column])[not_missing(frame[id_column])]) if id_column else set()
    if id_column:
        ID_SETS[role] = ids
    controlled_categoricals={}
    for semantic in ('timepoint','laterality','volume_unit','baseline_diagnosis','trace_state','qc'):
        field=resolved(fields, semantic)
        if field:
            controlled_categoricals[semantic] = safe_categories(frame[field])
    SOURCE_PROFILES[role]={
        'file_present':True,
        'status':'AUDITED_REAL_FILE',
        'filename':name,
        'rows':int(len(frame)),
        'columns':int(len(frame.columns)),
        'headers':list(map(str,frame.columns)),
        'field_resolution':fields,
        'nonmissing_completeness':counts,
        'wide_mri_column_candidates':wide_mri_headers(frame.columns),
        'distinct_participants':display_count(len(ids)) if id_column else None,
        'rows_beyond_first_per_participant':display_count(int(len(frame)-len(ids))) if id_column else None,
        'observed_labels':controlled_categoricals,
    }
    print(role, '| rows=',len(frame),'| fields resolved=',sum(x['status']=='EXACTLY_ONE' for x in fields.values()),
          '| unresolved=',[k for k,v in fields.items() if v['status']!='EXACTLY_ONE'])

longitudinal | rows= 258 | fields resolved= 0 | unresolved= ['participant_id', 'scan_id', 'series_or_acquisition_id', 'acquisition_date', 'timepoint', 'region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference', 'trace_reason']
repeat_pairs | rows= 90 | fields resolved= 3 | unresolved= ['scan_id', 'series_or_acquisition_id', 'region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference', 'trace_reason']
landmark12 | rows= 199 | fields resolved= 0 | unresolved= ['participant_id', 'scan_id', 'series_or_acquisition_id', 'acquisition_date', 'timepoint', 'region', 'laterality', 'volume_value', 'volume_unit', 'baseline_diagnosis', 'extraction_model', 'model_version', 'qc', 'trace_state', 'trace_rule_version', 'trace_reference', 'trace_reason']


In [21]:
CROSS_FILE = {'participant_overlap':{}, 'joinability':{}, 'unverified_reuse_warnings':[]}
for left,right in (('longitudinal','landmark12'),('longitudinal','repeat_pairs'),('landmark12','repeat_pairs')):
    pair=f'{left}__{right}'
    if left in ID_SETS and right in ID_SETS:
        l,r=ID_SETS[left],ID_SETS[right]
        CROSS_FILE['participant_overlap'][pair] = {
            'both_distinct':display_count(len(l & r)),
            'left_only':display_count(len(l - r)),
            'right_only':display_count(len(r - l)),
            'source_id_namespace_agreement':'NOT_PROVEN_BY_STRING_MATCH',
        }
        CROSS_FILE['joinability'][pair]='SAME_FIELD_GROUP_DETECTED_CHECK_RAW_ID_NAMESPACE_BEFORE_JOIN'
    else:
        CROSS_FILE['joinability'][pair]='BLOCKED_MISSING_OR_AMBIGUOUS_REAL_PARTICIPANT_KEY'

for role, frame in DATA.items():
    fields=SOURCE_PROFILES[role]['field_resolution']
    pid=resolved(fields,'participant_id')
    sid=resolved(fields,'scan_id')
    aq=resolved(fields,'series_or_acquisition_id')
    time=resolved(fields,'timepoint')
    if pid and sid:
        part=frame[[pid,sid]].copy()
        part=part[not_missing(part[pid]) & not_missing(part[sid])]
        multi_person_scan=(part.drop_duplicates().groupby(sid)[pid].nunique()>1).sum()
        CROSS_FILE['joinability'][role+'_study_key']={
            'candidate_study_id':sid,
            'distinct_study_ids':display_count(part[sid].nunique()),
            'ids_assigned_multiple_people':display_count(multi_person_scan),
            'status':'REVIEW' if multi_person_scan else 'NO_CROSS_PERSON_COLLISION_IN_OBSERVED_ROWS',
        }
    if pid and time:
        pair=frame.loc[not_missing(frame[pid]) & not_missing(frame[time]),[pid,time]].drop_duplicates()
        CROSS_FILE['joinability'][role+'_observed_timepoint_pairs']={
            'distinct_person_timepoint_pairs':display_count(len(pair)),
            'participants_with_2plus_raw_timepoints':display_count((pair.groupby(pid)[time].nunique()>=2).sum()),
        }
    if pid and aq:
        CROSS_FILE['joinability'][role+'_acquisition_id']={
            'candidate_acquisition_column':aq,
            'nonnull_rows':int(not_missing(frame[aq]).sum()),
        }
print(json.dumps(CROSS_FILE,indent=2))

{
  "participant_overlap": {},
  "joinability": {
    "longitudinal__landmark12": "BLOCKED_MISSING_OR_AMBIGUOUS_REAL_PARTICIPANT_KEY",
    "longitudinal__repeat_pairs": "BLOCKED_MISSING_OR_AMBIGUOUS_REAL_PARTICIPANT_KEY",
    "landmark12__repeat_pairs": "BLOCKED_MISSING_OR_AMBIGUOUS_REAL_PARTICIPANT_KEY",
    "repeat_pairs_observed_timepoint_pairs": {
      "distinct_person_timepoint_pairs": 90,
      "participants_with_2plus_raw_timepoints": 16
    }
  },
  "unverified_reuse_warnings": []
}


In [22]:
LONG_PAIR_CHECK = {
    'status':'NOT_EVALUABLE',
    'note':'This is a source-data structural readiness check, not a final eligible MCI cohort or ETL result.',
}
if 'longitudinal' in DATA:
    frame=DATA['longitudinal']
    fields=SOURCE_PROFILES['longitudinal']['field_resolution']
    needed=('participant_id','timepoint','region','laterality','volume_value','volume_unit')
    mapping={field:resolved(fields,field) for field in needed}
    if all(mapping.values()):
        named=frame.rename(columns={actual:semantic for semantic,actual in mapping.items()}).copy()
        named['region'] = clean_series(named['region']).str.lower()
        named['laterality'] = clean_series(named['laterality']).str.lower()
        named['timepoint'] = clean_series(named['timepoint']).str.lower()
        # Only literal source labels below are recognized; unmatched values are not silently recoded.
        tmap={'bl':'baseline','baseline':'baseline','m00':'baseline','m0':'baseline',
              'm12':'12m','12m':'12m','12month':'12m','12-month':'12m'}
        lmap={'left':'left','right':'right','lh':'left','rh':'right'}
        named['recognized_timepoint']=named['timepoint'].map(tmap)
        named['recognized_laterality']=named['laterality'].map(lmap)
        hip=named[named['region'].eq('hippocampus')].copy()
        valid=hip[hip['recognized_timepoint'].notna() & hip['recognized_laterality'].notna()
                  & not_missing(hip['volume_value']) & not_missing(hip['volume_unit'])
                  & not_missing(hip['participant_id'])]
        observed=valid.groupby('participant_id').apply(
            lambda part: {
                (t,s) for t,s in zip(part['recognized_timepoint'],part['recognized_laterality'])
            }, include_groups=False)
        complete={'baseline','12m'}
        sides={'left','right'}
        paired=sum(all((t,s) in seen for t in complete for s in sides) for seen in observed)
        LONG_PAIR_CHECK={
            'status':'DESCRIPTIVE_STRUCTURE_CHECK_ONLY',
            'recognized_hippocampal_rows':int(len(valid)),
            'participants_with_observed_bilateral_baseline_and_12m':display_count(paired),
            'raw_source_region_hippocampus_rows':int(len(hip)),
            'omitted_or_unrecognized_hippocampal_rows':int(len(hip)-len(valid)),
            'source_scan_id_available':bool(resolved(fields,'scan_id')),
            'acquisition_date_available':bool(resolved(fields,'acquisition_date')),
            'trace_reliability_evidence_available':all(resolved(fields,f) for f in ('trace_state','trace_rule_version','trace_reference')),
            'note':'Eligibility is not asserted: provenance, valid units, MCI, scan linkage and landmark-window checks must also pass.'
        }
    else:
        LONG_PAIR_CHECK['missing_or_ambiguous_real_columns']=[k for k,v in mapping.items() if v is None]
print(json.dumps(LONG_PAIR_CHECK,indent=2))

{
  "status": "NOT_EVALUABLE",
  "note": "This is a source-data structural readiness check, not a final eligible MCI cohort or ETL result.",
  "missing_or_ambiguous_real_columns": [
    "participant_id",
    "timepoint",
    "region",
    "laterality",
    "volume_value",
    "volume_unit"
  ]
}


In [23]:
AUDIT_REPORT_PATHS = [
    RUN_BASE / 'ichi2027_neuroimaging_phase01' / 'shareable_output' / 'ICHI2027_Phase01_Audit_SHAREABLE.json',
    DRIVE_ROOT / 'phase01_outputs' / 'ICHI2027_Phase01_Audit_SHAREABLE.json',
    DRIVE_ROOT / 'ICHI2027_Phase01_Audit_SHAREABLE.json',
    RUN_BASE / 'ICHI2027_Phase01_Audit_SHAREABLE.json',
]
PHASE01 = None
PHASE01_DIGEST = None
for candidate in AUDIT_REPORT_PATHS:
    if candidate.is_file():
        loaded=json.loads(candidate.read_text(encoding='utf-8'))
        if loaded.get('phase','').startswith('01'):
            PHASE01=loaded
            PHASE01_DIGEST=sha256_file(candidate)
            break

UPSTREAM = None
if PHASE01:
    UPSTREAM={name:{'status':entry.get('status'),'commit':entry.get('commit'),
                    'freeze_reference':entry.get('freeze_reference')}
              for name,entry in PHASE01.get('repositories',{}).items()}
    print('Original Phase 01 audit found. Upstream commit evidence carried forward.')
else:
    print('Original Phase 01 audit JSON not found. No upstream commit IDs will be inferred.')

Original Phase 01 audit JSON not found. No upstream commit IDs will be inferred.


In [24]:
SCIENTIFIC_GATES = {
    'actual_three_source_files_present':len(DATA)==3,
    'phase01_audit_loaded':PHASE01 is not None,
    'original_tfl_v15_verified':bool(PHASE01 and PHASE01.get('gates',{}).get('exact_original_tfl_v15_tag_verified')),
    'imagewg_ddl_discovered':bool(PHASE01 and PHASE01.get('gates',{}).get('schema_ddl_discovered')),
    'longitudinal_participant_key_resolved':bool('longitudinal' in SOURCE_PROFILES and
        resolved(SOURCE_PROFILES['longitudinal'].get('field_resolution',{}),'participant_id')),
    'landmark_participant_key_resolved':bool('landmark12' in SOURCE_PROFILES and
        resolved(SOURCE_PROFILES['landmark12'].get('field_resolution',{}),'participant_id')),
    'longitudinal_study_key_resolved':bool('longitudinal' in SOURCE_PROFILES and
        resolved(SOURCE_PROFILES['longitudinal'].get('field_resolution',{}),'scan_id')),
    'longitudinal_acquisition_date_resolved':bool('longitudinal' in SOURCE_PROFILES and
        resolved(SOURCE_PROFILES['longitudinal'].get('field_resolution',{}),'acquisition_date')),
    'original_trace_state_plus_reference_available':bool('longitudinal' in SOURCE_PROFILES and all(
        resolved(SOURCE_PROFILES['longitudinal'].get('field_resolution',{}), field)
        for field in ('trace_state','trace_rule_version','trace_reference'))),
}
SCIENTIFIC_GATES.update({
    'actual_four_arm_adapters_audited':False,
    'omop_concepts_verified':False,
    'fhir_r4_bundle_validation_executed':False,
    'micdm_target_load_executed':False,
    'cohort_equivalence_analysed':False,
    'comparative_etl_performance_measured':False,
})

SHAREABLE_REPORT={
    'project':'ICHI2027_neuroimaging_fhir_omop_fidelity',
    'phase':'02_real_source_reconciliation',
    'run_utc':UTC_NOW,
    'scientific_results_status':'NOT_RUN: this phase checks actual source evidence and freezes unresolved mappings',
    'privacy':'Schema and aggregates only; no participant rows, actual identifiers or source CSV bytes.',
    'environment':{'python':sys.version.split()[0], 'pandas':pd.__version__, 'platform':platform.platform()},
    'phase01_audit_sha256':PHASE01_DIGEST,
    'upstream_revisions_from_phase01':UPSTREAM,
    'input_profiles':SOURCE_PROFILES,
    'cross_file_evidence':CROSS_FILE,
    'longitudinal_pair_structure':LONG_PAIR_CHECK,
    'gates':SCIENTIFIC_GATES,
    'actual_ETL_comparator_scores':None,
    'actual_MRI_FHIR_OMOP_measurement_results':None,
}

PRIVATE_REPORT={
    'run_utc':UTC_NOW,
    'source_file_fingerprints':FINGERPRINTS,
    'local_input_paths':{role:str(source_files[name]) for name,role in SOURCE_ROLES.items() if name in source_files},
    'phase01_input_audit_digest':PHASE01_DIGEST,
    'privacy':'Private local reproducibility manifest; never commit to a public GitHub repository.',
}

public_path = OUTPUT_DEST / 'ICHI2027_Phase02_SHAREABLE.json'
local_public_path = SHAREABLE_LOCAL / public_path.name
private_path = PRIVATE_OUTPUT / 'ICHI2027_Phase02_PRIVATE_fingerprints.json'
public_text=json.dumps(SHAREABLE_REPORT,indent=2,ensure_ascii=False)
public_path.write_text(public_text,encoding='utf-8')
local_public_path.write_text(public_text,encoding='utf-8')
private_path.write_text(json.dumps(PRIVATE_REPORT,indent=2,ensure_ascii=False),encoding='utf-8')

print(json.dumps({'phase02':'COMPLETE','real_input_files':len(DATA),
                  'shareable_audit':str(public_path),
                  'shareable_sha256':sha256_file(public_path),
                  'private_fingerprint_manifest':str(private_path),
                  'scientific_findings':'NOT GENERATED',
                  'unresolved_gates':[key for key,ok in SCIENTIFIC_GATES.items() if not ok]},indent=2))

{
  "phase02": "COMPLETE",
  "real_input_files": 3,
  "shareable_audit": "/content/drive/MyDrive/NeuroFHIR_ICHI2027/phase02_outputs/ICHI2027_Phase02_SHAREABLE.json",
  "shareable_sha256": "ff586daf2cf21d4e6d2e26b3acaa869f2b3321a80e37502a88590ad49a57a14e",
  "private_fingerprint_manifest": "/content/ichi2027_neuroimaging_phase02/private_outputs/ICHI2027_Phase02_PRIVATE_fingerprints.json",
  "scientific_findings": "NOT GENERATED",
  "unresolved_gates": [
    "phase01_audit_loaded",
    "original_tfl_v15_verified",
    "imagewg_ddl_discovered",
    "longitudinal_participant_key_resolved",
    "landmark_participant_key_resolved",
    "longitudinal_study_key_resolved",
    "longitudinal_acquisition_date_resolved",
    "original_trace_state_plus_reference_available",
    "actual_four_arm_adapters_audited",
    "omop_concepts_verified",
    "fhir_r4_bundle_validation_executed",
    "micdm_target_load_executed",
    "cohort_equivalence_analysed",
    "comparative_etl_performance_measured"
  ]


In [25]:
if DRIVE_AVAILABLE:
    print('Google Drive save confirmed:',str(public_path), '| bytes:',public_path.stat().st_size)
else:
    try:
        from google.colab import files
        files.download(str(local_public_path))
    except ImportError:
        print('Shareable Phase 02 JSON:',local_public_path)
    except Exception as exc:
        print('Download not available; shareable file retained:',local_public_path,type(exc).__name__)

Google Drive save confirmed: /content/drive/MyDrive/NeuroFHIR_ICHI2027/phase02_outputs/ICHI2027_Phase02_SHAREABLE.json | bytes: 14376


In [26]:

import base64, zlib
backup_bytes=zlib.decompress(base64.b64decode('eNrNPf1z27ixv99fwea9KaVXhrGdS5NTT+0otpKo51geScn1np6KoShIYkORDEnZ8Tnu3/52FwAJkqDsXHIzzZwlCgQWi8V+4+Nuv7Me+TwMs0c9a/6dZd3Cnyhh+U3CofSRH6/4I4fK+Sfu7/Mgjpgf76Mc3kb7MBTvghVWfu49f3HsP3suG+x47q283INXt3eiKN7nyT6n/haiJIv3qc8lAghpl8RpbmU3mWNl+2WSxj7Psv+LBEyoX5S56T7qzKGiKzDzliF3LPvxzobPJEjw6/FH/AyiLPfCkMq9aOVl/f6Je+I+tReO5W+5/6E/S/e8+wh7WMAHIfu1xPCP/JOTH5brryHGOo13VuLl2zBYWpIyl/CzJAfVALg8D3ZcVVG/HQs/f40jXmvgx2HIfUQ/U21OcRg8LSvK8q2XYe+OrAePrhqKY/0riyPHimGqktDL13G6c6wUuoVJaQASlLfgv2RVviyf3s1O2cX4Z6tfoO9G8XVHjcDd537XDbIYe/HyTrdsOXl3wV4OpkNoisTpxMAR0VWQxpG74XnHHp2+GZ0cnTxnP48nP7HJeDxDTnjixzDgKLetYC3alUXYEVsFaadr8TDjVpanHazi+terThf+lZ1fTsZ/H57OCCwgUODyBNjO3wbUb8T3aRzsvE0QbVgCFOVHJ7YOYvR+MBuy0cXlO4RRAQlwkjS4ApKwIAJ2MbQbv5sdbigZTWs6fTOYDAcvz4fsfHw6ODe0zbZeylGkDK1hCiygDnBQnN5YQWR1KkNwrCpmjlXrrtsrYeFHAcvdfUCqJ9B1lGcklY7FPwVZzmIppCbWGf7jElAfnsEwOlXI9mwyOB2ywdnF6JiFcbQJ8v0qiLyQnY7fXp4PAUdA7Nz1syvbOdA05Qn3cpZ4QZo9rEUI7L7z0g/HJ8b62jim43cTaDYZnw+nyP2Bn3d+DZKOGpVjdWwddWReHR/8XfZmV7jzDCZiqHizyuRPVsAd/MnbmzP6vkAeffVmNGFKXOwGnMH7wegcpxGAvfJAMMoKeXpTm1RSNJs43oTcBX3jFQqM+q3WHU1hPgAywMVJLl/yTz5PcmtELYdpGqe91pY1jDQFtC6q1Vo3sSZ2RATdHSrEOrmA1sD8PgeGoPd96rTbhNEkWDkVruDvQsdUW8shD+kLNDRqTCgz4AnijRi+JhpbNI1WFOcWYcZXPUAWLVcHWnddxiJvxxkzCpCQU3Y2nM4qmJISEfqqUAOoLuujIyVZE3Ij9C8ScDE+tDHuar9Lss6tQMbu2YAPaKeQCYMJzzBLfhAGHlLMdmwwFnZPWhSnSbnyn03TKgfHeIQab2X3agN0bFkBkLd7aA20MXXvwDhGKxhQ/6T7jZ2I5y+eP33+/cnR1zgRK762QJufPPszWwch76A/4VjLMAa/5/jo5Pv/wY+6St4CH0jT74rGdT69DvKtUCkIsOvGCY86drq0u8ixW1BIITcwLdoOmP4czUYADkcn9HbLldeTLVyY1lWHkOsCkrbd7Zmnb+vuE3QUkJfyGmopz/dpBFW2/NMq2PCs4i6UT29H0+no4jWbjX8aXqDuvbVRmUYefT4RX15EX+CC0Pde+JH76AN4J5F49K68IETOETVzqyy4K3vDeYjAeWFbGCIMPOef8vroJOYpd8HP7aT2/J/e41+PHv+wQMjwh6xH7Vx4AhvRdcP4GmB1jePDHv2QexHLeBrwrCO+WjoVL13gkTDyOkB518tIf0BX1J/qs60rGDnbBVkGXs7hnv5tQop6kKMB1RhEner0tHa7CjLwPm+EDHWuvBB1yi4AoQKx6z+r4wDqi+pYQWZdwKwa+EuieVFxnPFDNOxbgerIzHhr+8dbhcAdKcwj60fZ+McCNaE1qdQ0tMH5CPxI4sqam4EcH/gBONQ5C0BZgX9QK3LshKegN+UzaEWuXqSiKKcvYLJ/gefF9B/lE5XCR7fu52S+F6me1TO0Av/kRjyir8v35SPblzUoGIt8vtcbyTKmtykfUeWJ2tDZ3ogR8RCLU+b5H/fAPaRVFYripYBAz3vtud55DUClQGumD6OBjg4DlRSh0SgU41HP/JO30x6LKkQg+UMQplIrl2CosIEIRk9JDOxKGJS/HPsKURHfaJHKpxP1ksnyDGJtYVX5Fae2BRzwcpc8bHab8g22wD7lI5Ax8vJ4F/hgtYsyEPm9DyKD3SxTDwSjeAUBE3yuwcuF1+S7NLuBmJOnXhjkN9SV9tOxtxxUUbLlBDsLaCRh2oRxFYd78ItIDAmKeHLki+KB7XZPyx8+/dhxLwPsdihc1WbiV1tn+ygQE0IPTqXYoUJF+72/3+nlEP7KTpvAl+AahUEE/BF4myjOgoz6KH4xCBhUHaR3s7pjrz6xZUjfWk0qBNkKNs1OwRalHmUS2A5QDqlLehJzBrSP1/m1p+YQ2WmDBCNPjamXxM4EKU5RPYkMD4bLRY1G16KXK1B0iteqJVrXZdE65SD+6ZqnWmESJIIUqqjR10efOoAvBz4YiD6J4Mc9sRvDGCGNw/LlPjPIIgyPy6Ykjdpv4HfAwFsGBE6UtQBI9yGvjNpQXAVXedUGlQNJOOgzHaQqE3qwXtGxQZYDEFuayUrl1k68rIq0KKihqwr1OkIgdLh3bf5AEOHsrgMerrKOj6IVNRyR5Q1DVwyN610tbAX/FFkdI230USUAg5cgYbgZz6Fbbx/mHd29U0DAjZ0vwJdKwD9eoS9VvunWu95HqxaUMr4D2x74jhUnMm8XWdJFcMGP3mUdk6eMMpcByAzCZw69c8ARiggkTJ5HYBRELKS3UKYGhwk0fVTUSAypawh7Be9Dh7Z03sj7QZdYYEIujz38x+B0dv4LG18M6X0IkQO971r9vnUsaw3evhy9fjd+N7VNUQSQaq6IsiDXXcpdT3wjd0arAM0ilBH0O7OzhqDaeCnlWRxeAeUEOznWB37TcCiB+oCAqEEEg0oOzKLZOcTqcx25xfxogVTAFyJhqTQIUqNKLKJM1S+tInwNlo7t0kDOV9aR33WkwScEYxkB3immZ3YJRoapvQ2SJPa9XfIZVHScboPo8y5YYVz2N8AuidPP6AKkgf8ZrD+IZ+pFnzv//ExhyqIb+Fcd8fj5v7s2poLdkZkIih9BGrZdYrwtcp1EFqkhEQTp8lJ/K2u2hzmZtwarDPTcxFpQAXGA90kr7jcDXbDUAQqdKSSZGyKaRbU5BR3I8ATHJbsvIpGss0rjJPKMKSLJ9KJi1/prDc/2mOQW3YCP0Ad5XsDZGIpooLrg5dAblu2TBNg3w3SGnccxA1m50fqwG+KQgQprUT/E0dSF0InY1wG1I4eHk4YC07V+7FtPTyzgeeSJNcTQOy+HWYUAd/D4f0WIazH3Se9Pjxe3x87TkzsZ61LrlgyAQHiuqi0od6qHgvRpUFIoRIdhYrif7Jdh4As6M6Qk+r6LRobyi+bGssUjA/UqXOee6PHu26aNnj194b3gP3xV2mg6HExO37Cz0QRj0Hl1xNUUIS1MZPV0uFoKaSm+b4Xk2MYa3j7fArv+yldy+aMGrbrwUL5bmBRFqXSRzdEm9OaLitlTOfe76mLHOg5BL2EFjSrNvALaOVG1yO0a+AydxSCq85AJC1NjNQS0OAKrJ9TOKIRFbcSHkn9twlSSZo7QFspfKcq792TWSbhBw3SQCHVoxtF1H7pggIhnTWpde5jlvodeOhGyuWiyaKOBcWK0TPuYHCXwC0uutPZJGHuUaxfADfpGVOErchJgKK4o6HSN6CL9VIsvRlTR2UFxR9IoSAeUNfltYD7VChG277pmnpI4UnVEFCVCW77qtafa23GmJRee5bQIqa+EPqF+WpHANi6tHcC4iP30LDe+7Vp/aGSxkTBdPTF8AOfUC8DdmoCODXacVp86a/s0jtZgGGAwG0sGVRn40pi3vrGknQA9i6jf2d324brXKcwJWAJgTIHVvZKJMAvJpAEaFq94WEy6SWTligkxYs3YYxmGsjS8xkvFWVYIDmIRgpTYtXCYMolFo74VwpR1cJnTRZEH252Rp5C40tvudIVLmGAHRcNu16hv6/AfrnBhAnlKkk1jTXqVRRINhZ2X0Oq/0+ise2d06zDM0sELtzDDnQPIkcemZZEK7aXWBLN7awstL1adErl0CF6eQNbuCTa+E9g64hciXelfTs3iobTRmaTAhZRDnQIQu7RZhuqQamPuVLcIPLFPxxevzkenM0z4k/hPmazi4gqg3ZXygqsf+ppgtRd9Ja7mozVl2Z7Oxpc9K7+ONZ5GgnhAv1WwpoRGXgp5vLbyLQelBHZMUIjGiPRxrSnnltxvYSl0wDFEA+bqakCyrU7hngnTV/DmIs5fYXRaQRdXohQq+TaFbjc82gdQqG0/sE6n74WxsYBhyvWoCirSqg38fE+5DnCvZBsiB60gk6zqyHabAN6KCEmHAC3nRaKhbvhJdLFMWhAd/OIbL6CenPgv/ux5z77GEz4bzAY1bfgK2HQ4AQa9mE1rr6Q9vJyMX43Ejg797eiMTYeNNk01e8CtOUA9g2KpoTPXzbUQ7UUflAxKkQpyehSxOkVCReVyaCOLEM568HhIzYa839QnddcPyvrJilZ/mZ9dkVg71grnvA+aDxMvPGEyz8ZUVO1YPAJ2AObr2/t8/fgFuGHADEsAgvncrG9zFB0bjdY12/FdnN60xeOEgyvTfaBcEpBgFIJO10W31uQnCEl9j8pdeQdCRovWJIcCpsiA9UTc0fAMRPaIlha17GUFp7pCi0N02ZrzWYvUgHnnWBdVOMGrvtc5uagnGIIS9j1h9G3yU+welrvIGB1csGUZ+MClLdINKH7UzWORLDmU4dQSmwaSU96AXOlGak4BMLvVsmGrR42Y1RKKqGl3CWak42sZzesJISLmXMJdADn2O7DwB3eb1Pae1KF3/n0Avuqg7nKAvBB/9OsEqS8K11me2A1dlUrqS/RaQF1Uk2CNt4supS5VgdwuyfNOU8CKSoY5kGqxYEBArs46tMIS8lWRwvJBjvv3cVJ10bOyQlhddTMuh1WXaD76xt0oRO7+F7Ij1ToQ3hnGWuHOer5TTAxBXdR6rFsAonH/ttl3zQrQ5ixDrcIsDN6djcA8CbOAsG2nBaimSQw1SgFA55lGYpQjWyrCWt1CPxrbyJSy3SNPBl15NCcPaEi0ZDSt5GKg7qOZNdQFTSElhGEuPeQ5B+sDnQq9YmpSJOoFEkxfrWgk8avomsCtYHRB5OOW0ELoM4wO9IQokgwkq2uSWVxUaJsctuQ3cbQCxZ6Czk/APmm91DupzuPjos8v6TRegjYCYSoyqS0iUWtb0wXCPUV2B/v12cKR9MGjLXBzoFDaXSW98B617Ke54vJFv19dgUEF80kEWLTeU4R2RsUPPewjDfj8g0inO1caCBmd0c6ksuM/VDv+1o7x0XP+4sVy9fRrHOPTyXg6Jckne6nbmxiiptBL7B5As+x/gf6Vy7uyZB9BhQAIgCK2zzi79tII5Afmer6opR1Cvs6dNNhsyd/t1HdDVzY/O423la3S9L6sXn/bXCUL0v7avkUM7hi7JSTubEPQvybclHdPCx0KYVloUPahk/aV5UMQC6ewg9jYEK6XFJ8byb2YI8qL5qaxgiGXcb5lSluYFERo/dFKW/0YG/FkcRTemNs+PtSWRtXaOIXGYXtjGUOABiFHOkHL7G0g/t2RtbIvxjO0ce+HF+zlL2w6m2DE8nYwO31jMkt3h2mrM6ykad+eDt4O4f3w/Iy9nozfXbKz4YwCNHb6Znj6E5sMfmYwgRdQbXqJofjL4avxZMj+Ph5d1HimZQnqXhReno9Pf4IOVUQ2nrBimVzY4cvBZDY6HV0OLmbsp+EvdluQKdQiKUJkUgwTWjJ4Qkv1jW7EvGkma1ybBKsv9UwzUxO1z7BW1/toqNq2GbDWFj3DZuvSX2y6sDAYkWgOTL4bjqovHLE51HSg1mIBVju5MaX6qTZ+VBxsKoDG4Ff/0Wq8QIimLN4OguKAyT2fSKk+1XdxBZoVwShMrbtJ432yvOkApC7140Yipdfp/vVYhhhfwpfIBX+ymdil+IFDUb9N8xQeDit2ivYAjTZ5LzwaVbvhzhQkKcfQqj2gOfOAkpsILA6RK0Ffl8fwVYfboGa7TlKu8GT4fjT8mTazNFrL7SwXYybIeDmcTMcXuGp2PpqO4Gl0wcYvp8PJe/Slxz9PH6CvNF5EfjUyI5ou8hvD2DcEcSYmE28Q4qLrEA/T86LJSL+BRwqnrtw+KsxuO8uUXq0gZ72lwYrgi3Y20L1jhicY2EkS7jOWetcl8AZcAlqIToKiQ3TRZKd/ciAB0D553sfeb6BkTa09ROT0JsIDt3vexzYyQTyD/uLBxIf3cfGwITeO8ZTjc36vczPeEf9+7Z98Vdr3fAwG9nIwmgjzbtiIX4g/+h7D94Pzd3hYqHEsEGiHZ5Zm2yDDQweeTNk+pjVatQXaCy1MfmLWMhOHkh3K8Hq4kuPhYYFgEyxDbr09HVl+vMWFcTDjw9m53LTi2ubNkcBtVZdYmfueKQdL6cJq/cWD3IFam/sdg4jzFcREhrMLesam2A9uSt1U91yLTE5NM0HAn2B6+FYkXOrWnr7EUh89Im0EYk2Z9cKwI8GVUV/LnsuVVL0ppT7ULtT+rUdrLb0iQ1XZ2umphRiFtXLI7rqtXgR1NldkWtQ3stVeVw7ZtALTSN0GUK/yMKDlpLbB1GrcA/K/rDHEEFYYEBJqLU7kCiz4iK9p1QtPBW4i3KXxF4jCVeZYTB1VoIU4TE/lAA1r40aJZnc5TEf/1l6GIMSGDfTV0t3RUb2g8rstPWzvIB7t2ccnO2iDn9pzHOVb7fdjvcAQyIQCX4zUoBJ9OSL2gp/iG+Rpq73bFi/uDrCYoibTZqrfnDwXk2xIs+5DYOmc1DdwF0ELzdC2QSKb1Pjc5R875V5W3Erb7ofTxs0+VJ7jX8swXWCVyOugv1SvVUFWVju0BlD1uAhaRZ01nDK9Cqm4xZfCr2lYYwihvLO+2MiqnJ1GqIbbT8KbjhkDcZaTQpteWwaCVv9zJxNaFx5Q4eF5d3LnW+jvNF5WNJC5JzqX64f7FWc0nKz1tLZK2qKUl5LbIl54mChT8uW0yw06jUBPdJHQdIgx44It55EYu9hLK7oWpoDeI/yuNA1QFXfoy8kxoF7zUlpdwcJXORtOTyejyxlu4JzOJu9OZ+8mQ5nBGF+c/2K3Zm9K4hei5ek+IrrfxD1f4n8XUQE4uWJKi0NKDLxXRqqwHvchYduzTODLF+fCUR8wTRPU0IU37cjGuyDPAbM4ZZjEfcDoEdzjB5BBoidzGqw8LtxbxnHYaU9+tEKsHy68H2bjOGI7cHUQpzyjw6+ASfGwj9YNMnnDv5K+lVyKqyynGc8t1Q8ZtSMlveoh+cWEFTrX5DJnwFK4fcVK0vgKPDAA5MiN/nSIzkEn2rEoOAdG++BtOEVjKjP8+BriEnAiyA3PIKLPAGiYxSDSWeba9wZ4Ldm9mqjObbVmhJmq3TLY7GNkULzXQK10LfrVFYOaY0hLBuog8+JAyFXr+neLu/wf/vz82bOjr1pVoGVFNhlejiczwHn2xrD//Ms2kdfvsqHC4lagS1GTDSB2yVlxi4XYduYc2vcu+yivxvg2YH8TCJ0iXwxAY53LNwDk6BhIXj1mJMvZ2ei1uCWk+hp5tNybjls4GrPY3Mb0sO3pYi9rn7gZn7NyU7rYL0RbAusbgbrmdXe5G5qOWYnLRBy86QB0UppnaJA6NjBN26ZgSYR+fXutoY4kVF/fE2PaTK83XcJwPhjvxroESz0cvG2QHa+uEv3VMFYt+uKYQ3k6Do9v3VSPmTk2OCE7PJKsvZRF9+1noQO1v1bOj2pAGi+7d23giv3F1Bw5SA5MQMKdlGCsxMkl5xYiYhUbm9Z87bE6wUkSYAE/eygC4rCfa71LIL7kHl6EhqO0lDkDjkyhhxVic+2lq8p2SYNWv6e3v0/HF/JQCHV7EVv7Ws+js8y6DsIQwlax9Qs8G+j1Gx8HOlo/5cfHy686DnQ6Gl7MRq9Gp+z1YGa8l0LkLxhtSmX6lr9yTwn6Rphi6vb7T+tpMqVNiXZMiJjdU/pI2nbDjgFbnadl+TpkV8fPmFpalq6PAoEmvsJUG9puQdxEv/knGAJrgMu9TQmy4ZCISxmuN2y1CnFh1ce12C/vOwNnY+fVgTR7q1xipgdnH/gNU+6X7LyZ96tvTvWilemooXTi7snxSRmv5/lwTM0lPsPlDcLZun8Q5Xr9txhCCe1rB6DPRLEA9p80B61hQxX3RuzwHzSE+6MUTfzL8IKJVZ3iToJ6NPSgIVHG9/cZl8g2d80HuIpM9G+MmFpuSahrcHWBVosaB5sFYUm6Y2ArEgjMM6GYkSvE3uv6NOziBA9i4LV1maaCjZXX2yBl6fdsucdLvxjFZuIeCWHNWhvuAn+1A42cAnHJRtzXQCyVMA5sBL0IZoDpuckOtIDIHm+1uOKM5yGuPdKdo9hUXr5iaHpnPAZT3oknPOF+4y6nNKZLl3ql014JY4hQRNo1uChi+cNkN++/E69+Rc8+YgduyQPdgXdGwRz6TKwvZayy3AXxRs/KcUmL+lexslzDkCn5wq9CaRLOYKZtiFPhbNYcE56k8W+grynZRSGOm03KyW5auIfoL+AQWJqWpg1+jsIAO0b08ToBzKQJhHA7Pu1kdxtdyitjxW6iWzu5ybe46RPvF5bC5mYJTECnOz9aFDcJ271k5TIljozhC3kVLryST656AHf1sNejdtJXg4h6G+VEwszgTU1xBNFTGu+KmLenvP+Gq4IndMAbi8kts3s1DdaQhDTORGBWJHzsnrZ6e9g1AcYt73fq1TMQtbbCHerVdZRj1k3D2TlTYhpDN36cYmuTcyhbvJ2MGN1uOn47vmT6xU2Suxutjcdz1Km1Fmm+V6hKp5jpB/RAnWjHMJp0xUuz5Nx5+Raq3+JOADoS2Dxa0y1jKTojUjujq+eO1EkjHUgrhwoEBJ+Kc4b38WkpxZfyZBwNBY/FpfFq76u8HejWYA3wQKI5SJKKjfLY8ixxDYP1Osjf7JdWEQbetC13a0kwcYEDUgyCFf2K02aK5KSeHCnBCOpXgdWvT35iae9rR6fVJcyyae3SZhMqqkrlCGcNK9kfpj76Wp6vbm+KRB+E1cjxzAO9HjTWQXTstWOWWidOI79ygEK/AYZOpZaTnlXZOziyQz3df7/sES7EqvuhaQ9CIX5SbxZh7OG7Zcu8I0mNPMNb0unBzU0HrB4OR9FW4yemRE7ipFH/PqRKvwDgrcSWbfQIrNfDi+EEJujMPgyhtP9Mav05xE3FNTPxB9JYdWdV11pxbsUfFne/Xxb7xH/69MXxV+0eat6SbM4dVW5vzrwrcXo4SHd0/LbBMHiigBwYeKlLXPVI3uGslfnW6y+5fkMd7MzcVXwd0WUWiGlDEXTNd1u3X+etEWaquF9l1U4on4ZnkhvdfOUN2mdyEFbl7t6/WIUA0mDxmh8viMSx6DoKjuGm7ZIpv7OQSyqM9B1yKJJZ/rAeYSvkvsIenOJrBgZhggpoKmz0pOLWu0FyE6m8XrnaRXyppOHRB55GwAYJ94u+1IKq6vOSvF1LrduEXrTZext6JRxh+ULVF4VPHxV9qBagJ9dxY0gShiAFNngULcX/xwHefq//ZrsgilMoffbd3f8DYURXOA=='))
backup_folder=DRIVE_ROOT/'notebooks' if DRIVE_AVAILABLE else PROJECT_ROOT/'notebook_backup'
backup_folder.mkdir(parents=True,exist_ok=True)
backup_path=backup_folder/'ICHI2027_Colab_02_Real_Source_Reconciliation.ipynb'
if backup_path.exists() and hashlib.sha256(backup_path.read_bytes()).digest()!=hashlib.sha256(backup_bytes).digest():
    backup_path=backup_folder/('ICHI2027_Colab_02_Real_Source_Reconciliation_'+hashlib.sha256(backup_bytes).hexdigest()[:12]+'.ipynb')
backup_path.write_bytes(backup_bytes)
print('Clean code-only notebook backup:',str(backup_path))

Clean code-only notebook backup: /content/drive/MyDrive/NeuroFHIR_ICHI2027/notebooks/ICHI2027_Colab_02_Real_Source_Reconciliation.ipynb
